# RAG Pipeline Demo — Pháp luật & Tin tức Ma tuý Việt Nam

**Day 8 | RAG Pipeline v2**

Notebook này demo end-to-end pipeline:
1. Semantic Search (Task 5)
2. Lexical Search / BM25 (Task 6)
3. Hybrid Retrieval + Reranking (Task 7, 9)
4. Generation có Citation (Task 10)
5. So sánh kết quả A/B

In [ ]:
import sys
from pathlib import Path

# Đảm bảo project root trong sys.path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print('✓ Setup complete. Project root:', PROJECT_ROOT)

## 1. Kiểm tra dữ liệu đã index

In [ ]:
import json
import numpy as np

INDEX_DIR = PROJECT_ROOT / 'data' / 'index'
chunks = json.loads((INDEX_DIR / 'chunks.json').read_text(encoding='utf-8'))
embeddings = np.load(INDEX_DIR / 'embeddings.npy')
meta = json.loads((INDEX_DIR / 'meta.json').read_text(encoding='utf-8'))

print(f'📊 Index stats:')
print(f'  Chunks: {len(chunks):,}')
print(f'  Embeddings: {embeddings.shape}')
print(f'  Embedding model: {meta["embedding_model"]}')
print(f'  Chunk size: {meta["chunk_size"]}, overlap: {meta["chunk_overlap"]}')

# Breakdown by type
from collections import Counter
types = Counter(c['metadata']['type'] for c in chunks)
print(f'\n  Type breakdown: {dict(types)}')

## 2. Task 5 — Semantic Search Demo

In [ ]:
from src.task5_semantic_search import semantic_search

query = 'hình phạt tội tàng trữ ma tuý'
results = semantic_search(query, top_k=5)

print(f'🔍 Semantic Search: "{query}"')
print(f'  {len(results)} kết quả\n')
for i, r in enumerate(results, 1):
    src = r['metadata'].get('source', 'unknown')
    print(f'  [{i}] score={r["score"]:.4f} | {src}')
    print(f'       {r["content"][:120]}…\n')

## 3. Task 6 — BM25 Lexical Search Demo

In [ ]:
from src.task6_lexical_search import lexical_search

query = 'Điều 249 tàng trữ trái phép chất ma tuý'
results = lexical_search(query, top_k=5)

print(f'📖 BM25 Lexical Search: "{query}"')
print(f'  {len(results)} kết quả\n')
for i, r in enumerate(results, 1):
    src = r['metadata'].get('source', 'unknown')
    print(f'  [{i}] score={r["score"]:.4f} | {src}')
    print(f'       {r["content"][:120]}…\n')

## 4. Task 7 — Reranking Demo (RRF + Cross-Encoder)

In [ ]:
from src.task7_reranking import rerank_rrf, rerank
from src.task5_semantic_search import semantic_search
from src.task6_lexical_search import lexical_search

query = 'ca sĩ chi dân bị bắt vì ma tuý'

dense = semantic_search(query, top_k=8)
sparse = lexical_search(query, top_k=8)

# RRF merge
merged = rerank_rrf([dense, sparse], top_k=5)
print(f'🔀 After RRF merge ({len(dense)} dense + {len(sparse)} sparse → {len(merged)} merged):')
for i, r in enumerate(merged, 1):
    print(f'  [{i}] rrf_score={r["score"]:.5f} | {r["metadata"].get("source","")} | {r["content"][:80]}…')

# Cross-encoder rerank
print('\n🎯 After Jina cross-encoder rerank:')
reranked = rerank(query, merged, top_k=3, method='cross_encoder')
for i, r in enumerate(reranked, 1):
    print(f'  [{i}] score={r["score"]:.4f} | {r["metadata"].get("source","")} | {r["content"][:80]}…')

## 5. Task 9 — Full Retrieval Pipeline

In [ ]:
from src.task9_retrieval_pipeline import retrieve

test_queries = [
    'Hình phạt tội tàng trữ trái phép chất ma tuý theo pháp luật VN',
    'Ca sĩ nào bị bắt vì liên quan ma tuý 2024 2025',
    'Cai nghiện bắt buộc thủ tục như thế nào',
]

for q in test_queries:
    print(f'\n🔎 Query: {q}')
    results = retrieve(q, top_k=3)
    for i, r in enumerate(results, 1):
        print(f'  [{i}] score={r["score"]:.4f} [{r["source"]}] {r["metadata"].get("source","")} | {r["content"][:80]}…')

## 6. Task 10 — Generation với Citation

In [ ]:
from src.task10_generation import generate_with_citation, reorder_for_llm, format_context

# Demo reorder_for_llm
sample_chunks = [
    {'content': f'Chunk {i+1}', 'score': 1.0 - i*0.1, 'metadata': {'source': f'doc{i}.md'}}
    for i in range(5)
]
reordered = reorder_for_llm(sample_chunks)
print('📋 Reorder for LLM (Lost-in-Middle prevention):')
print('  Input:  ', [c['content'] for c in sample_chunks])
print('  Output: ', [c['content'] for c in reordered])
print('  → Chunk 1 (best) ở ĐẦU, Chunk 2 ở CUỐI, Chunk 5 (worst) ở GIỮA')

In [ ]:
# Full generation với citation
question = 'Hình phạt cho tội tàng trữ trái phép chất ma tuý theo pháp luật Việt Nam?'

print(f'Q: {question}\n')
result = generate_with_citation(question, top_k=5)

print('A:', result['answer'])
print(f'\n[{len(result["sources"])} chunks | via {result["retrieval_source"]}]')

In [ ]:
# Câu hỏi về tin tức
question2 = 'Những nghệ sĩ nào đã bị bắt vì liên quan tới ma tuý theo các bài báo?'
print(f'Q: {question2}\n')
result2 = generate_with_citation(question2, top_k=5)
print('A:', result2['answer'])
print(f'\n[{len(result2["sources"])} chunks | via {result2["retrieval_source"]}]')

## 7. So sánh A/B: Hybrid+Rerank vs Dense-Only

In [ ]:
from src.task5_semantic_search import semantic_search
from src.task9_retrieval_pipeline import retrieve

q = 'cai nghiện bắt buộc theo Nghị định 163/2026'

print(f'Query: "{q}"\n')

# Config B: Dense only
dense_results = semantic_search(q, top_k=3)
print('Config B (Dense-only):')
for r in dense_results:
    print(f'  score={r["score"]:.4f} | {r["metadata"].get("source","")} | {r["content"][:80]}…')

# Config A: Hybrid + Rerank
hybrid_results = retrieve(q, top_k=3, use_reranking=True)
print('\nConfig A (Hybrid + Rerank):')
for r in hybrid_results:
    print(f'  score={r["score"]:.4f} [{r["source"]}] | {r["metadata"].get("source","")} | {r["content"][:80]}…')

## 8. Kiến trúc hệ thống

```
Data Collection
  ├── Task 1: DOCX (Luật PCMT 2025, NĐ 163/2026, NĐ 28/2026)
  └── Task 2: JSON news (VnExpress, Ngôi Sao) via Crawl4AI
         ↓
Task 3: MarkItDown → Markdown files (data/standardized/)
         ↓
Task 4: RecursiveCharacterTextSplitter (500/50)
      + all-MiniLM-L6-v2 (384-dim)
      → 1,934 chunks → Weaviate Cloud + local NPY cache
         ↓
Retrieval Pipeline (Task 9)
  ├── Task 5: Semantic Search (cosine sim on local index)
  ├── Task 6: BM25 Lexical Search (rank-bm25)
  ├── Task 7: RRF merge → Jina Reranker v2 (cross-encoder)
  └── Task 8: PageIndex fallback (vectorless, score < 0.005)
         ↓
Task 10: reorder_for_llm → format_context → GPT-4o-mini
      → Answer with [Citation, Year]
         ↓
app.py: Streamlit Chat UI (conversation memory + source display)
```